# Entrenar el VAE en Colab Pro (desde la extensión "Colab" de VSCode)

Este notebook corre el pipeline de `scripts/preprocess_vae.py` + `scripts/VAE.py`
contra un runtime remoto de **Google Colab Pro**, conectado directamente desde
VSCode con la extensión oficial **"Colab"** (publisher: Google) — sin salir del
editor. Guía completa en el `README.md`, sección *"Alternativa: entrenar el VAE
en Colab Pro desde VSCode"*.

**Antes de correr este notebook:**
1. Instala la extensión **Colab** en VSCode (`Ctrl+Shift+X` → buscar "Colab").
2. Inicia sesión con la cuenta de Google que tiene Colab Pro.
3. En el selector de kernel (arriba a la derecha), elige el runtime **Colab**
   (no un intérprete local) y en `Runtime > Change runtime type` selecciona GPU.

Luego corre las celdas de este notebook en orden.

In [ ]:
import os

REPO_URL = "https://github.com/Tamaracarrasco/AS4501-Proyect.git"
REPO_DIR = "/content/AS4501-Proyect"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull


## Dependencias

`requirements.txt` fija una build **CPU** de torch (`torch==2.12.0+cpu`), pensada
para correr localmente. Colab ya trae preinstalado un torch con soporte CUDA
funcionando con la GPU del runtime — si lo pisamos con la versión CPU del
`requirements.txt`, perdemos la GPU. Por eso instalamos todo *excepto* las líneas
de torch/torchvision.

In [ ]:
!grep -v -i '^torch' requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt


## Verificar GPU

`VAE.py` detecta CUDA automáticamente (no hay que tocar el código ni el CONFIG).

In [ ]:
!nvidia-smi

import torch
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Datos: montar Drive

Los archivos de `/data/` (el tar de stamps y el CSV de features) pesan varios GB
y no están en GitHub — se suben una vez a Google Drive y se montan en cada
sesión. Ajusta `DRIVE_DATA_DIR` a la carpeta real dentro de tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- AJUSTA esta ruta a donde tengas los datos en tu Drive ---
DRIVE_DATA_DIR = "/content/drive/MyDrive/AS4501-Proyect/data"

os.environ["COSMOS_TAR"] = f"{DRIVE_DATA_DIR}/cosmos_TC_202602.tar.gz.part_aa"
os.environ["COSMOS_FEATURES"] = f"{DRIVE_DATA_DIR}/features_images_20260707.csv"

print(os.environ["COSMOS_TAR"])
print(os.environ["COSMOS_FEATURES"])


## Salidas persistentes en Drive

El runtime de Colab es efímero (se borra al desconectar). `preprocess_vae.py`
y `VAE.py` escriben siempre en `<repo>/file_out_data/` (sin flag para cambiar la
ruta) — para no perder el `.npz`, los checkpoints y las figuras entre sesiones,
apuntamos esa carpeta a Drive con un symlink, hecho una sola vez por sesión.

In [ ]:
DRIVE_OUT_DIR = "/content/drive/MyDrive/AS4501-Proyect/file_out_data"
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

LOCAL_OUT_DIR = f"{REPO_DIR}/file_out_data"
if os.path.islink(LOCAL_OUT_DIR) or os.path.exists(LOCAL_OUT_DIR):
    !rm -rf {LOCAL_OUT_DIR}
!ln -s {DRIVE_OUT_DIR} {LOCAL_OUT_DIR}

print(f"{LOCAL_OUT_DIR} -> {DRIVE_OUT_DIR}")


## Preprocesamiento → `file_out_data/vae_input.npz`

Aplica los filtros de catálogo, la corrección por extinción galáctica, la
normalización "otro" (signed sqrt scaling) y el split train/val/test
estratificado. Con la GPU de Colab esto sigue corriendo en CPU (es I/O y NumPy),
pero solo hay que hacerlo una vez por conjunto de datos: el `.npz` queda en
Drive y se reutiliza entre sesiones.

In [ ]:
# prueba rápida (carga solo 300 stamps, para validar que el pipeline corre):
# !python scripts/preprocess_vae.py --n-max 300

# corrida completa:
!python scripts/preprocess_vae.py


## Entrenamiento del VAE

Corre con la config por defecto (`z_dim=64`, 200 épocas) o con overrides por
CLI. `run_id` se auto-incrementa mirando `vae_summary*.json` en `file_out_data/`
(en Drive), así que corridas de sesiones distintas no se pisan entre sí.

In [ ]:
!python scripts/VAE.py --epochs 200 --z-dim 64

# overrides comunes:
# !python scripts/VAE.py --epochs 100 --z-dim 32 --beta-final 0.5
# !python scripts/VAE.py --wandb                    # dashboard en vivo (requiere login de wandb)
# !python scripts/VAE.py --sweep                     # barrido con scripts/parameters_vae.dat


## Visualizar el último resultado

Muestra la figura de reconstrucción más reciente de la última corrida (`run_id` más alto).

In [ ]:
import glob
from IPython.display import Image, display

run_dirs = sorted(
    glob.glob(f"{DRIVE_OUT_DIR}/figures/vae*"),
    key=lambda p: int(p.rsplit("vae", 1)[-1]) if p.rsplit("vae", 1)[-1].isdigit() else -1,
)

if run_dirs:
    latest_run = run_dirs[-1]
    recon_imgs = sorted(glob.glob(f"{latest_run}/recon_e*.png"))
    if recon_imgs:
        print(f"corrida: {latest_run}")
        display(Image(filename=recon_imgs[-1]))
    else:
        print(f"sin figuras de reconstrucción todavía en {latest_run}")
else:
    print("todavía no hay corridas en", DRIVE_OUT_DIR)


## Siguientes pasos

- Todo lo relevante (`vae_metrics{N}.csv`, `vae_summary{N}.json`, `figures/vae{N}/`,
  `checkpoints/vae_best{N}.pt`, `vae{N}.input`) ya quedó en
  `DRIVE_OUT_DIR` gracias al symlink — no se pierde al cerrar el runtime.
- Para compartirlo con el equipo vía git: copia (o descarga) esos archivos desde
  Drive a tu clon local de `file_out_data/` y comitea desde ahí — evita comitear
  directo desde el runtime efímero de Colab.